In [2]:
import numpy as np
import pandas as pd
import xarray as xr

In [3]:
# static features
stat_dict = {}
stat_features = ['clelev', 'clfaotxt', 'cltveg', 'phh2o', 'topsoil_bulk_density', 'vegetation_type_11', 'wetlandtype', 'climatetype']
stat_files = ['clelev.nc', 'clfaotxt.nc', 'cltveg.nc', 'phh2o.nc', 'topsoil_bulk_density.nc', 'vegetation_type_11.nc', 'wetlandtype.nc', 'climatetype.nc'] 
for i, feat in enumerate(stat_features):
    filename = f'../../data/TEM-MDM/{stat_files[i]}' 
    print(filename)
    feature = xr.open_dataset(filename)
    feature = feature[feat].values
    stat_dict[feat] = feature

../../data/TEM-MDM/clelev.nc
../../data/TEM-MDM/clfaotxt.nc
../../data/TEM-MDM/cltveg.nc
../../data/TEM-MDM/phh2o.nc
../../data/TEM-MDM/topsoil_bulk_density.nc
../../data/TEM-MDM/vegetation_type_11.nc
../../data/TEM-MDM/wetlandtype.nc
../../data/TEM-MDM/climatetype.nc


In [4]:
# non-spatial features
nosp_features = ['ch4','co2']
nosp_files = ['ch4-1979-2018','kco21979-2018']
ch4_dict, co2_dict = {}, {}
with open('../../data/TEM-MDM/ch4-1979-2018.txt', 'r') as f:
    for line in f:
        year, ch4 = line.split()
        ch4_dict[int(year)] = float(ch4)
with open('../../data/TEM-MDM/kco21979-2018.txt', 'r') as f:
    for line in f:
        year, co2 = line.split()
        co2_dict[int(year)] = float(co2)

In [5]:
def convert2year(data):
    days_in_month = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    mask = np.zeros(data.shape, dtype=bool)
    for month, days in enumerate(days_in_month):
        mask[:, :, month, :days] = True 
    valid_data = data[mask]
    valid_data = valid_data.reshape(data.shape[0], data.shape[1], 365)
    return valid_data

In [6]:
site_info = pd.read_csv('../../data/FLUXNET-CH4/FLUXNET_CH4_2024.csv')
fluxnet_data = pd.read_csv('../../data/FLUXNET-CH4/FLUXNET_T1_DD.csv')

In [7]:
location = {}
for index, line in site_info.iterrows():
    location[line['SITE_ID'].replace('-','.')] = (line['LOCATION_LAT'], line['LOCATION_LONG'], line["LOCATION_ELEV"], line["IGBP"], line["MAT"], line['MAP'])

In [8]:
fluxnet_data['TIMESTAMP'] = pd.to_datetime(fluxnet_data['TIMESTAMP'])
fluxnet_data['Year'] = fluxnet_data['TIMESTAMP'].dt.year
fluxnet_data['TIMESTAMP'] = pd.to_datetime(fluxnet_data['TIMESTAMP'])

fluxnet_data = fluxnet_data[~((fluxnet_data['TIMESTAMP'].dt.month == 2) & (fluxnet_data['TIMESTAMP'].dt.day == 29))]
grouped = fluxnet_data.groupby(['Site', 'Year'])

In [9]:
splitting_year_dict = {}
for site, group in fluxnet_data.groupby('Site'):
    unique_years = sorted(group['Year'].unique())
    midpoint = (len(unique_years)-1) * 6 // 7
    splitting_year = unique_years[midpoint] if unique_years else None
    splitting_year_dict[site] = splitting_year

print(splitting_year_dict)

{'AT.Neu': np.int32(2011), 'BR.Npw': np.int32(2015), 'BW.Gum': np.int32(2018), 'BW.Nxr': np.int32(2018), 'CA.SCB': np.int32(2016), 'CA.SCC': np.int32(2015), 'CH.Cha': np.int32(2015), 'CH.Dav': np.int32(2016), 'CH.Oe2': np.int32(2018), 'CN.Hgu': np.int32(2016), 'DE.Dgw': np.int32(2017), 'DE.Hte': np.int32(2017), 'DE.SfN': np.int32(2013), 'DE.Zrk': np.int32(2017), 'FI.Hyy': np.int32(2016), 'FI.Lom': np.int32(2009), 'FI.Si2': np.int32(2015), 'FI.Sii': np.int32(2017), 'FR.LGt': np.int32(2017), 'HK.MPM': np.int32(2017), 'ID.Pag': np.int32(2016), 'IT.BCi': np.int32(2017), 'IT.Cas': np.int32(2009), 'JP.BBY': np.int32(2017), 'JP.Mse': np.int32(2012), 'JP.SwL': np.int32(2016), 'KR.CRK': np.int32(2017), 'MY.MLM': np.int32(2014), 'NL.Hor': np.int32(2008), 'NZ.Kop': np.int32(2014), 'PH.RiF': np.int32(2013), 'RU.Ch2': np.int32(2015), 'RU.Che': np.int32(2015), 'RU.Cok': np.int32(2014), 'RU.Fy2': np.int32(2017), 'SE.Deg': np.int32(2017), 'UK.LBT': np.int32(2013), 'US.A03': np.int32(2017), 'US.A10': n

In [10]:
def nearest_patch_point(lat, lon):
    lat_index = round((lat + 90) * 2) 
    lon_index = round((lon + 180) * 2)  

    return (lat_index, lon_index)

In [11]:
site_wet = {
    "CA.SCB": 2,
    "CA.SCC": 1,
    "DE.Hte": 2,
    "DE.Zrk": 2,
    "DE.SfN": 2,
    "FI.Lom": 2,
    "FI.Si2": 2,
    "FI.Sii": 2,
    "FR.LGt": 2,
    "JP.BBY": 2,
    "NZ.Kop": 2,
    "RU.Ch2": 2,
    "RU.Cok": 2,
    "SE.Deg": 2,
    "US.A03": 2,
    "US.ICs": 2,
    "US.A10": 2,
    "US.Atq": 2,
    "US.Beo": 2,
    "US.Bes": 2,
    "US.NGB": 2,
    "US.BZB": 1,
    "US.BZF": 2,
    "US.Uaf": 1,
    "US.Ivo": 2,
    "US.Los": 2,
    "US.NGC": 2,
    "BR.Npw": 3,
    "BW.Gum": 4,
    "BW.Nxr": 5,
    "ID.Pag": 3,
    "MY.MLM": 3,
    "US.NC4": 3,
    "US.DPW": 4,
    "US.LA2": 4,
    "US.Myb": 4,
    "US.ORv": 3,
    "US.OWC": 4,
    "US.Sne": 4,
    "US.Tw1": 4,
    "US.Tw4": 4,
    "US.Tw5": 4,
    "US.WPT": 4
}


In [12]:
# temporal
train_data_x, train_data_y, test_data_x, test_data_y = [], [], [], []
test_site_list = []
for (site, year), group in grouped:
    if site not in location.keys():
        continue
    lat, lon, elev, IGBP, mat, map = location[site]
    if group.shape[0] != 365:
        continue
    if IGBP != 'WET':
        continue
    temp = group.to_numpy()[:,-6:-3]
    temp = temp.astype(dtype=float)
    lat_ind, lon_ind = nearest_patch_point(lat, lon)
    for i, feat in enumerate(stat_features):
        if i == 0:
            input = np.tile(elev, (365))[..., np.newaxis]
        elif feat == 'wetlandtype':
            if site in site_wet.keys():
                wet = site_wet[site]
            else:
                wet = stat_dict[feat][lat_ind, lon_ind]
            wet = np.tile(wet, (365))[..., np.newaxis]
            input = np.concatenate((input, wet), axis=1)
        else:
            new_input = stat_dict[feat][lat_ind, lon_ind]
            new_input = np.tile(new_input, (365))[..., np.newaxis]
            input = np.concatenate((input, new_input), axis=1)
    ch4_value = np.tile(ch4_dict[year], (365))[..., np.newaxis]
    co2_value = np.tile(co2_dict[year], (365))[..., np.newaxis]
    input = np.concatenate((input, ch4_value), axis=1)
    input = np.concatenate((input, co2_value), axis=1)

    input = np.concatenate((input, temp[:,2][..., np.newaxis]), axis=1) #prec

    # solr 
    input_file = f'../../data/TEM-MDM/daily_ecmwf_SOLR_{year}.nc'
    feature = convert2year(xr.open_dataset(input_file, engine="netcdf4")['SOLR'].values[lat_ind, lon_ind,:,:].reshape(1,1,12,31))
    input = np.concatenate((input, feature.reshape(365,1)), axis=1)

    input = np.concatenate((input, temp[:,0][..., np.newaxis]), axis=1) # tair
    
    T = group['TA_F'].to_numpy()
    VPD = group['VPD_F'].to_numpy()
    vp = (0.6108 * np.exp(17.27*T/(T+237.3)))*10 - VPD
    input = np.concatenate((input, vp[..., np.newaxis]), axis=1) # vapr

    # npp 
    input_file = f'../../data/TEM-MDM/monthly_NPP_{year}.nc'
    feature = xr.open_dataset(input_file, engine="netcdf4")['NPP'].values[lat_ind, lon_ind,:].reshape(1,1,12)
    feature = np.nan_to_num(feature, nan=0.0)
    feature = np.repeat(feature[:, :, :, np.newaxis], 31, axis=3)
    feature = convert2year(feature)
    input = np.concatenate((input, feature.reshape(365,1)), axis=1)
    
    if np.isnan(input).any():
        print('Input nan')
        continue
    label = group['FCH4_F'].to_numpy()
    mask = np.isnan(label)
    label[mask] = group['FCH4_F_ANNOPTLM'].to_numpy()[mask]
    if year > splitting_year_dict[site]:
        test_data_x.append(input)
        test_data_y.append(label)
        test_site_list.append(site)
    else:
        train_data_x.append(input)
        train_data_y.append(label)

Input nan
Input nan
Input nan
Input nan
Input nan
Input nan
Input nan
Input nan
Input nan
Input nan
Input nan
Input nan
Input nan
Input nan
Input nan
Input nan
Input nan
Input nan
Input nan
Input nan


In [13]:
train_data_x = np.stack(train_data_x, axis=0)
train_data_y = np.stack(train_data_y, axis=0)
test_data_x = np.stack(test_data_x, axis=0)
test_data_y = np.stack(test_data_y, axis=0)

In [14]:
train_data_x.shape, train_data_y.shape, test_data_x.shape, test_data_y.shape

((71, 365, 15), (71, 365), (17, 365, 15), (17, 365))

In [16]:
np.save('../../processed_data/FLUXNET-CH4/temporal/train_data_x.npy', train_data_x)
np.save('../../processed_data/FLUXNET-CH4/temporal/train_data_y.npy', train_data_y)
np.save('../../processed_data/FLUXNET-CH4/temporal/test_data_x.npy', test_data_x)
np.save('../../processed_data/FLUXNET-CH4/temporal/test_data_y.npy', test_data_y)

In [17]:
num_folds = 5
folds = {0: ['BW.Gum', 'FI.Sii', 'RU.Ch2', 'US.Ivo', 'US.Srr', 'US.Bes'],
    1: ['CA.SCB', 'FI.Lom', 'US.WPT', 'US.BZF', 'US.LA2', 'US.BZB'],
    2: ['DE.Zrk', 'RU.Che', 'US.MRM', 'US.LA1', 'US.Los', 'US.NC4'],
    3: ['JP.BBY', 'FR.LGt', 'US.ICs', 'US.Atq', 'US.Myb', 'US.OWC'],
    4: ['DE.SfN', 'DE.Hte', 'US.ORv', 'US.DPW', 'FI.Si2', 'US.Beo']}

In [18]:
for fold in range(num_folds):
    test_sites = set(folds[fold]) 
    data_x, data_y = [], []
    for (site, year), group in grouped:
        if site not in test_sites:
            continue
        lat, lon, elev, IGBP, mat, map = location[site]
        if group.shape[0] != 365:
            continue
        if IGBP != 'WET':
            continue
        temp = group.to_numpy()[:,-6:-3]
        temp = temp.astype(dtype=float)
        lat_ind, lon_ind = nearest_patch_point(lat, lon)
        for i, feat in enumerate(stat_features):
            if i == 0:
                input = np.tile(elev, (365))[..., np.newaxis]
            elif feat == 'wetlandtype':
                if site in site_wet.keys():
                    wet = site_wet[site]
                else:
                    wet = stat_dict[feat][lat_ind, lon_ind]
                wet = np.tile(wet, (365))[..., np.newaxis]
                input = np.concatenate((input, wet), axis=1)
            else:
                new_input = stat_dict[feat][lat_ind, lon_ind]
                new_input = np.tile(new_input, (365))[..., np.newaxis]
                input = np.concatenate((input, new_input), axis=1)
        ch4_value = np.tile(ch4_dict[year], (365))[..., np.newaxis]
        co2_value = np.tile(co2_dict[year], (365))[..., np.newaxis]
        input = np.concatenate((input, ch4_value), axis=1)
        input = np.concatenate((input, co2_value), axis=1)

        input = np.concatenate((input, temp[:,2][..., np.newaxis]), axis=1) #prec

        # solr 
        input_file = f'../../data/TEM-MDM/daily_ecmwf_SOLR_{year}.nc'
        feature = convert2year(xr.open_dataset(input_file, engine="netcdf4")['SOLR'].values[lat_ind, lon_ind,:,:].reshape(1,1,12,31))
        input = np.concatenate((input, feature.reshape(365,1)), axis=1)

        input = np.concatenate((input, temp[:,0][..., np.newaxis]), axis=1) # tair
        
        T = group['TA_F'].to_numpy()
        VPD = group['VPD_F'].to_numpy()
        vp = (0.6108 * np.exp(17.27*T/(T+237.3)))*10 - VPD
        input = np.concatenate((input, vp[..., np.newaxis]), axis=1) # vapr

        # npp 
        input_file = f'../../data/TEM-MDM/monthly_NPP_{year}.nc'
        feature = xr.open_dataset(input_file, engine="netcdf4")['NPP'].values[lat_ind, lon_ind,:].reshape(1,1,12)
        feature = np.nan_to_num(feature, nan=0.0)
        feature = np.repeat(feature[:, :, :, np.newaxis], 31, axis=3)
        feature = convert2year(feature)
        input = np.concatenate((input, feature.reshape(365,1)), axis=1)
        
        if np.isnan(input).any():
            continue
        label = group['FCH4_F'].to_numpy()
        mask = np.isnan(label)
        label[mask] = group['FCH4_F_ANNOPTLM'].to_numpy()[mask]
        data_x.append(input)
        data_y.append(label)
        print(site, year, label.shape, np.max(label))
    data_x = np.stack(data_x, axis=0)
    data_y = np.stack(data_y, axis=0)
    print("fold", fold, data_x.shape, data_y.shape, np.max(data_y))
    np.save(f'../../processed_data/FLUXNET-CH4/spatial/data_x_{fold}.npy', data_x)
    np.save(f'../../processed_data/FLUXNET-CH4/spatial/data_y_{fold}.npy', data_y)


BW.Gum 2018 (365,) 483.563322315711
FI.Sii 2013 (365,) 153.515458333333
FI.Sii 2014 (365,) 146.255791666667
FI.Sii 2015 (365,) 116.192875
FI.Sii 2016 (365,) 164.675625
FI.Sii 2017 (365,) 132.359145833333
FI.Sii 2018 (365,) 79.9650833333333
RU.Ch2 2014 (365,) 97.114675
RU.Ch2 2015 (365,) 82.3404791666667
RU.Ch2 2016 (365,) 81.35925
US.Ivo 2013 (365,) 42.0245416666667
US.Ivo 2014 (365,) 47.6107708333333
US.Ivo 2015 (365,) 36.2935273744241
US.Ivo 2016 (365,) 64.3158125
US.Srr 2014 (365,) 10.87000125
US.Srr 2015 (365,) 10.2566639583333
US.Srr 2016 (365,) 12.7582220833333
fold 0 (17, 365, 15) (17, 365) 483.563322315711
CA.SCB 2014 (365,) 106.205601875
CA.SCB 2015 (365,) 107.18146
CA.SCB 2016 (365,) 113.58094625
FI.Lom 2006 (365,) 127.904755666667
FI.Lom 2007 (365,) 217.510894104167
FI.Lom 2008 (365,) 170.345764395833
FI.Lom 2009 (365,) 184.903329416667
FI.Lom 2010 (365,) 131.315445520833
US.BZB 2014 (365,) 84.4521958333333
US.BZB 2015 (365,) 113.388708333333
US.BZB 2016 (365,) 229.113002083